In [77]:
# Basic Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

# PyTorch and Transformers
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
from transformers import logging
logging.set_verbosity_error()  # Suppress warnings from the Transformers library

# Utility
from tqdm import tqdm

In [78]:
import torch
import gc
torch.cuda.empty_cache()
gc.collect()

1861

In [79]:

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [80]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [81]:
class MultiTaskDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        text = row['text']
        labels = {
            "is_fake": row['is_fake'] if row['is_fake'] != -1 else None,
            "is_toxic": row['is_toxic'] if row['is_toxic'] != -1 else None,
            "is_hate_speech": row['is_hate_speech'] if row['is_hate_speech'] != -1 else None
        }
        encoded = self.tokenizer(
            text, 
            padding="max_length", 
            truncation=True, 
            max_length=self.max_length, 
            return_tensors="pt"
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0).to(device),
            "attention_mask": encoded["attention_mask"].squeeze(0).to(device),
            "labels": labels
        }

In [82]:
def train_model(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    task_losses = {"is_fake": 0, "is_toxic": 0, "is_hate_speech": 0}
    task_counts = {"is_fake": 0, "is_toxic": 0, "is_hate_speech": 0}

    for batch in dataloader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        fake_news_pred, toxicity_pred, hate_speech_pred = model(input_ids, attention_mask)
        
        loss = 0
        if labels["is_fake"] is not None:
            target_fake = torch.tensor(labels["is_fake"], device=device).float().view_as(fake_news_pred)
            loss_fake = criterion(fake_news_pred, target_fake)
            loss += loss_fake
            task_losses["is_fake"] += loss_fake.item()
            task_counts["is_fake"] += 1
        
        if labels["is_toxic"] is not None:
            target_toxic = torch.tensor(labels["is_toxic"], device=device).float().view_as(toxicity_pred)
            loss_toxic = criterion(toxicity_pred, target_toxic)
            loss += loss_toxic
            task_losses["is_toxic"] += loss_toxic.item()
            task_counts["is_toxic"] += 1

        if labels["is_hate_speech"] is not None:
            target_hate = torch.tensor(labels["is_hate_speech"], device=device).float().view_as(hate_speech_pred)
            loss_hate = criterion(hate_speech_pred, target_hate)
            loss += loss_hate
            task_losses["is_hate_speech"] += loss_hate.item()
            task_counts["is_hate_speech"] += 1
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_losses = {task: task_losses[task] / task_counts[task] for task in task_losses if task_counts[task] > 0}
    return total_loss / len(dataloader), avg_losses

In [83]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    task_losses = {"is_fake": 0, "is_toxic": 0, "is_hate_speech": 0}
    task_counts = {"is_fake": 0, "is_toxic": 0, "is_hate_speech": 0}
    predictions = {"is_fake": [], "is_toxic": [], "is_hate_speech": []}
    ground_truths = {"is_fake": [], "is_toxic": [], "is_hate_speech": []}
    accuracies = {}

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"]
            labels = batch["labels"]

            fake_news_pred, toxicity_pred, hate_speech_pred = model(input_ids, attention_mask)
            
            if labels["is_fake"] is not None:
                target_fake = torch.tensor(labels["is_fake"], device=device).float().view_as(fake_news_pred)
                loss_fake = criterion(fake_news_pred, target_fake)
                task_losses["is_fake"] += loss_fake.item()
                task_counts["is_fake"] += 1
                pred_fake_binary = (torch.sigmoid(fake_news_pred) > 0.5).cpu().numpy()
                predictions["is_fake"].extend(pred_fake_binary)
                ground_truths["is_fake"].extend(target_fake.cpu().numpy())

            if labels["is_toxic"] is not None:
                target_toxic = torch.tensor(labels["is_toxic"], device=device).float().view_as(toxicity_pred)
                loss_toxic = criterion(toxicity_pred, target_toxic)
                task_losses["is_toxic"] += loss_toxic.item()
                task_counts["is_toxic"] += 1
                pred_toxic_binary = (torch.sigmoid(toxicity_pred) > 0.5).cpu().numpy()
                predictions["is_toxic"].extend(pred_toxic_binary)
                ground_truths["is_toxic"].extend(target_toxic.cpu().numpy())

            if labels["is_hate_speech"] is not None:
                target_hate = torch.tensor(labels["is_hate_speech"], device=device).float().view_as(hate_speech_pred)
                loss_hate = criterion(hate_speech_pred, target_hate)
                task_losses["is_hate_speech"] += loss_hate.item()
                task_counts["is_hate_speech"] += 1
                pred_hate_binary = (torch.sigmoid(hate_speech_pred) > 0.5).cpu().numpy()
                predictions["is_hate_speech"].extend(pred_hate_binary)
                ground_truths["is_hate_speech"].extend(target_hate.cpu().numpy())
    
    avg_losses = {task: task_losses[task] / task_counts[task] for task in task_losses if task_counts[task] > 0}

    # Calculate metrics for each task
    metrics_table = []
    for task in ["is_fake", "is_toxic", "is_hate_speech"]:
        y_true = ground_truths[task]
        y_pred = predictions[task]

        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred,average="weighted", zero_division=0)
        recall = recall_score(y_true, y_pred,average="weighted", zero_division=0)
        f1 = f1_score(y_true, y_pred,average="weighted", zero_division=0)

        metrics_table.append({
            "Task": task,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1
        })

    # Convert metrics to DataFrame
    metrics_df = pd.DataFrame(metrics_table)
    return avg_losses, metrics_df

    # # Compute accuracy for each task
    # for task in predictions.keys():
    #     if ground_truths[task]:
    #         accuracies[task] = accuracy_score(ground_truths[task], predictions[task])
    
    # return avg_losses, predictions, ground_truths, accuracies

In [84]:
# MultiTaskModel Definition
class MultiTaskModel(nn.Module):
    def __init__(self, model_name, task_outputs):
        """
        Initializes the MultiTaskModel.
        
        Args:
            model_name (str): The name of the pretrained model to use (e.g., "bert-base-uncased").
            task_outputs (dict): A dictionary specifying the number of outputs for each task.
                                 Example: {"is_fake": 1, "is_toxic": 1, "is_hate_speech": 1}.
        """
        super(MultiTaskModel, self).__init__()
        # Load the pretrained transformer model
        self.shared_model = AutoModel.from_pretrained(model_name)
        self.task_heads = nn.ModuleDict({
            task: nn.Linear(self.shared_model.config.hidden_size, output_size)
            for task, output_size in task_outputs.items()
        })

    def forward(self, input_ids, attention_mask):
        """
        Forward pass of the model.
        
        Args:
            input_ids (torch.Tensor): Input token IDs.
            attention_mask (torch.Tensor): Attention masks for the input.

        Returns:
            Tuple of outputs for each task.
        """
        # Extract the shared representation from the transformer model
        shared_output = self.shared_model(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = shared_output.pooler_output  # Use the [CLS] token representation
        
        # Pass the shared representation to each task head
        task_outputs = {task: head(pooled_output) for task, head in self.task_heads.items()}
        
        return task_outputs["is_fake"], task_outputs["is_toxic"], task_outputs["is_hate_speech"]


In [85]:

df_combined = pd.read_csv("/home/s2shsinh/TWON_Metrics/dataset/augmented_annotations.csv")
train_data, val_data = train_test_split(df_combined, test_size=0.2, random_state=42)

   

In [86]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-german-cased")
train_dataset = MultiTaskDataset(train_data, tokenizer, max_length=128)
val_dataset = MultiTaskDataset(val_data, tokenizer, max_length=128)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Initialize model and optimizer
model = MultiTaskModel("google-bert/bert-base-german-cased", {"is_fake": 1, "is_toxic": 1, "is_hate_speech": 1}).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()



In [87]:
model

MultiTaskModel(
  (shared_model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [88]:
train_dataset

In [89]:
# Training loop
for epoch in range(3):
    train_loss, train_task_losses = train_model(model, train_loader, optimizer, criterion, device)
    val_task_losses, metrics_df = evaluate_model(model, val_loader, criterion, device)
    
    print(f"Epoch {epoch+1}, Train Loss: {train_loss}, Task Losses: {train_task_losses}")
    print(f"Validation Task Losses: {val_task_losses}")
    print("Validation Metrics:")
    print(metrics_df)

/tmp/ipykernel_2849787/3536440188.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_fake = torch.tensor(labels["is_fake"], device=device).float().view_as(fake_news_pred)
/tmp/ipykernel_2849787/3536440188.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_toxic = torch.tensor(labels["is_toxic"], device=device).float().view_as(toxicity_pred)
/tmp/ipykernel_2849787/3536440188.py:31: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_hate = torch.tensor(labels["is_hate_speech"], device=device).float().view_as(hate_speech_

Epoch 1, Train Loss: 1.4965652055740357, Task Losses: {'is_fake': 0.5584478378097216, 'is_toxic': 0.4432003989517689, 'is_hate_speech': 0.49491696737209956}
Validation Task Losses: {'is_fake': 0.46511812753816867, 'is_toxic': 0.3678943197777931, 'is_hate_speech': 0.4554421059945796}
Validation Metrics:
             Task  Accuracy  Precision    Recall  F1-Score
0         is_fake  0.780000   0.777475  0.780000  0.778000
1        is_toxic  0.848667   0.848916  0.848667  0.841602
2  is_hate_speech  0.791333   0.782489  0.791333  0.780670


/tmp/ipykernel_2849787/3536440188.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_fake = torch.tensor(labels["is_fake"], device=device).float().view_as(fake_news_pred)
/tmp/ipykernel_2849787/3536440188.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_toxic = torch.tensor(labels["is_toxic"], device=device).float().view_as(toxicity_pred)
/tmp/ipykernel_2849787/3536440188.py:31: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_hate = torch.tensor(labels["is_hate_speech"], device=device).float().view_as(hate_speech_

Epoch 2, Train Loss: 1.0447432050307592, Task Losses: {'is_fake': 0.38455811753869057, 'is_toxic': 0.27788001467784246, 'is_hate_speech': 0.382305073171854}
Validation Task Losses: {'is_fake': 0.4263339920880947, 'is_toxic': 0.3561146827810939, 'is_hate_speech': 0.4429870479284449}
Validation Metrics:
             Task  Accuracy  Precision    Recall  F1-Score
0         is_fake  0.817333   0.817987  0.817333  0.812168
1        is_toxic  0.852333   0.849670  0.852333  0.849277
2  is_hate_speech  0.804333   0.797337  0.804333  0.797530


/tmp/ipykernel_2849787/3536440188.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_fake = torch.tensor(labels["is_fake"], device=device).float().view_as(fake_news_pred)
/tmp/ipykernel_2849787/3536440188.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_toxic = torch.tensor(labels["is_toxic"], device=device).float().view_as(toxicity_pred)
/tmp/ipykernel_2849787/3536440188.py:31: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_hate = torch.tensor(labels["is_hate_speech"], device=device).float().view_as(hate_speech_

Epoch 3, Train Loss: 0.6462210896809896, Task Losses: {'is_fake': 0.22672812088082234, 'is_toxic': 0.1567076856394609, 'is_hate_speech': 0.2627852831532558}
Validation Task Losses: {'is_fake': 0.4646403988466618, 'is_toxic': 0.4154804267702585, 'is_hate_speech': 0.4588389918208122}
Validation Metrics:
             Task  Accuracy  Precision    Recall  F1-Score
0         is_fake  0.808667   0.817750  0.808667  0.810783
1        is_toxic  0.849667   0.852736  0.849667  0.850850
2  is_hate_speech  0.802667   0.815178  0.802667  0.806747
